# 🚴 Estimación Aerodinámica (CdA) y Resistencia (Crr) desde archivos .FIT

Este cuaderno analiza la telemetría de archivos `.fit` (potencia, velocidad, altitud, tiempo) y aplica el modelo físico de fuerzas para estimar el coeficiente aerodinámico $C_d A$ y el coeficiente de rodadura $C_{rr}$.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path('..').resolve()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src import cargar_fit, preprocesar_datos, estimar_cda
from src.fit_analyzer import graficar_analisis_fit
from config import DATA_DIR, OUTPUT_DIR

# Seleccionar archivo .fit de muestra
ruta_fit = DATA_DIR / 'i150002737.fit'
print(f'Leyendo archivo: {ruta_fit}')
df_raw = cargar_fit(ruta_fit)
df_raw.head()

## 1. Preprocesamiento de Telemetría
Calcula derivadas temporales (aceleración, pendiente, distancia) y filtra paradas o datos con poca señal.

In [ ]:
df_proc = preprocesar_datos(df_raw, min_speed_ms=3.0, min_power_w=30.0)
print(f'✅ {len(df_proc)} registros filtrados de alta calidad.')
df_proc[['timestamp', 'potencia', 'velocidad', 'aceleracion', 'pendiente']].describe()

## 2. Optimización y Ajuste del Modelo Físico
Minimiza el Error Cuadrático Medio ($MSE$) entre la potencia medida y la predicha por las fuerzas aerodinámicas, gravitatorias, de rodadura e inercia.

In [ ]:
# Configurar peso total (Ciclista + Bicicleta + Equipamiento)
masa_total = 80.0  # kg
resultado = estimar_cda(df_proc, masa_total=masa_total, rho=1.15)

print('='*45)
print('🏆 RESULTADOS DEL MODELO')
print('='*45)
print(f" • CdA Estimado:            {resultado['cda']:.4f} m²")
print(f" • Crr Estimado:            {resultado['crr']:.5f}")
print(f" • Error RMSE:              {resultado['rmse']:.2f} W")
print(f" • Éxito en optimización:   {resultado['success']}")
print('='*45)

## 3. Visualización y Curvas de Validación

In [ ]:
graficar_analisis_fit(df_proc, resultado, guardar_ruta=OUTPUT_DIR / 'cda_analisis.png', mostrar=True)